# 01 - Data Preparation

使用 PySpark 对 KuaiRec small_matrix 进行全量数据处理：
- 数据加载与 Schema 检查
- 数据清洗
- 标签生成（`watch_ratio > 2` → `label=1`）
- 基础统计 EDA
- 按用户采样（300用户），输出 Parquet 供后续使用

In [ ]:
import os
import sys

# 解决 PyTorch/LightGBM 与 PySpark 的 OpenMP 冲突
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
spark = SparkSession.builder \
    .appName("KuaiRec_DataPrep") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

## 1. 数据加载

In [ ]:
# Original data download site: https://kuairec.com/
DATA_DIR = "./data/KuaiRec/data"

df = spark.read.csv(f"{DATA_DIR}/small_matrix.csv", header=True, inferSchema=True)
df.printSchema()

In [ ]:
df.show(5, truncate=False)

## 2. 基础统计

In [ ]:
total_rows = df.count()
n_users = df.select('user_id').distinct().count()
n_videos = df.select('video_id').distinct().count()

print(f"Total interactions : {total_rows:,}")
print(f"Unique users        : {n_users:,}")
print(f"Unique videos       : {n_videos:,}")
print(f"Matrix density      : {total_rows / (n_users * n_videos):.2%}")

In [ ]:
# watch_ratio 分布统计
df.select(
    F.min('watch_ratio').alias('min'),
    F.max('watch_ratio').alias('max'),
    F.mean('watch_ratio').alias('mean'),
    F.percentile_approx('watch_ratio', 0.5).alias('p50'),
    F.percentile_approx('watch_ratio', 0.75).alias('p75'),
    F.percentile_approx('watch_ratio', 0.9).alias('p90'),
    F.percentile_approx('watch_ratio', 0.95).alias('p95'),
    F.percentile_approx('watch_ratio', 0.99).alias('p99'),
).show()

In [ ]:
# watch_ratio 分布可视化（采样避免内存压力）
sample_pdf = df.select('watch_ratio').sample(fraction=0.01, seed=42).toPandas()

fig, ax = plt.subplots(figsize=(6, 4))

ax.hist(sample_pdf['watch_ratio'].clip(0, 5), bins=50, edgecolor='black')
ax.set_title('watch_ratio Distribution (clipped at 5)')
ax.set_xlabel('watch_ratio')
ax.axvline(x=2.0, color='red', linestyle='--', label='threshold=2.0')
ax.legend()

plt.tight_layout()
plt.show()

## 3. 数据清洗

In [ ]:
# 检查缺失值
null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
])
null_counts.show()

In [ ]:
# 清洗规则：
# 1. watch_ratio 不为空
# 2. video_duration > 0（排除时长异常视频）
# 3. watch_ratio >= 0（排除异常负值）
df_clean = df.filter(
    F.col('watch_ratio').isNotNull() &
    F.col('video_duration').isNotNull() &
    (F.col('video_duration') > 0) &
    (F.col('watch_ratio') >= 0)
)

dropped = total_rows - df_clean.count()
print(f"Dropped {dropped:,} rows ({dropped/total_rows:.2%})")
print(f"Clean dataset: {df_clean.count():,} rows")

## 4. 标签生成

参考 KuaiRec 官方文档及相关文献（Batch-Adaptive Doubly Robust Learning, 2024），
将 `watch_ratio > 2` 定义为正样本（用户循环播放超过2遍，视为强满意信号）。

在 CVR 预测框架下：
- **Click** = watch 行为（曝光 → 播放）
- **Conversion** = `watch_ratio > 2`（播放 → 深度满意）

In [ ]:
LABEL_THRESHOLD = 2.0

df_labeled = df_clean.withColumn(
    'label',
    F.when(F.col('watch_ratio') > LABEL_THRESHOLD, 1).otherwise(0)
)

# 标签分布
label_dist = df_labeled.groupBy('label').count().orderBy('label')
label_dist.show()

pos_count = df_labeled.filter(F.col('label') == 1).count()
total_count = df_labeled.count()
print(f"Positive rate: {pos_count/total_count:.2%}")

In [ ]:
# 标签分布可视化
label_pdf = label_dist.toPandas()
plt.figure(figsize=(5, 4))
plt.bar(['Negative (0)', 'Positive (1)'], label_pdf['count'], color=['steelblue', 'coral'])
plt.title('Label Distribution')
plt.ylabel('Count')
for i, v in enumerate(label_pdf['count']):
    plt.text(i, v + 1000, f'{v:,}', ha='center')
plt.tight_layout()
plt.show()

## 5. 按用户采样

取前300个用户的全部交互记录（约100万行），保留用户行为的完整性。
相比随机行采样，按用户采样不会破坏用户历史序列，计算用户统计特征时不会出现偏差。

In [ ]:
N_USERS = 300

sampled_users = [
    row.user_id
    for row in df_labeled.select('user_id').distinct().orderBy('user_id').limit(N_USERS).collect()
]

df_sampled = df_labeled.filter(F.col('user_id').isin(sampled_users))

sampled_rows = df_sampled.count()
print(f"Sampled users       : {N_USERS}")
print(f"Sampled interactions: {sampled_rows:,}")
print(f"Positive rate       : {df_sampled.filter(F.col('label')==1).count()/sampled_rows:.2%}")

## 6. 保存结果

In [ ]:
OUTPUT_PATH = "./data/processed/interactions_sampled.parquet"

df_sampled.write.mode('overwrite').parquet(OUTPUT_PATH)
print(f"Saved to {OUTPUT_PATH}")

spark.stop()
print("Spark session stopped.")